# Methane ASL properties

**Evidence: numerical verification.** Evaluate the compiled pressure-property callback at 298.15 K and 100,000 Pa, methane vapor.


Gross and Sadowski (2001), Table 2
([paper](https://doi.org/10.1021/ie0003887)), supplies the methane coefficients
in `gross-2001-methane-ethane/3`. McBride, Zehe and Gordon (2002),
[NASA/TP-2002-211556](https://ntrs.nasa.gov/citations/20020085330), Appendix D,
supplies the NASA9 methane record in `nasa-glenn-2002-hda-species/1`.
The packet preserves the source gas constant, 8.314510 J/(mol K), and
100,000 Pa standard pressure; `ThermochemistryPacket` performs the native
gas-constant conversion. The first methane interval is 200–1000 K.
Zotero parent: `AQDUURW4`; PDF: `SYJLCAXZ`.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from epcsaft import (
    Parameters,
    Pure,
    ThermochemistryPacket,
    PropertyObservable,
    PressureActionDirection,
)
from epcsaft import _core
from epcsaft.eos import property_asl_record

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "ARCHITECTURE.yaml").is_file()
)
PACKETS = ROOT / "data" / "packets"


## Load the physical records

An ASL record serializes the installed model for an external function.
Locate `libepcsaft_asl.so` beside the installed `_core` extension.
This example needs Pyomo and IDAES, but no external Ipopt executable.


In [2]:
p = Parameters.from_json(
    PACKETS / "gross-2001-methane-ethane/3/parameters/parameters.json",
    components=["methane"],
)
thermal = ThermochemistryPacket(
    ROOT / "data/packets/nasa-glenn-2002-hda-species/1/parameters/nasa9-records.json"
).select(["methane"])
model = Pure(p, thermochemistry=thermal)


## Compare callback values and exact derivatives

Rows are molar density, total enthalpy and ln fugacity coefficient. Numerical
arguments are `(T / K, P / Pa, w / 1)` with pure-component weight w = 1.
Compare values, T/P gradients and TT/TP/PP Hessian entries with native actions
normalized by 1 K and 1 Pa increments. The criterion is 10⁻¹⁰ relative with
a 10⁻²² absolute floor. Hessians are packed column-wise as TT, TP, PP, Tw,
Pw, ww. Gradient units are observable-unit per K or Pa; second derivatives
divide by the two corresponding units. The complete arrays are printed.


In [3]:
from pyomo.environ import ExternalFunction
import idaes

text = property_asl_record(model)
fn = ExternalFunction(
    library=str(Path(_core.__file__).resolve().with_name("libepcsaft_asl.so")),
    function="epcsaft_pressure_property",
)
temperature_increment_K = 1.0
pressure_increment_Pa = 1.0
t, p = (
    PressureActionDirection(temperature=temperature_increment_K),
    PressureActionDirection(pressure=pressure_increment_Pa),
)
obs = [PropertyObservable.TotalEnthalpy, PropertyObservable.LogFugacityCoefficient]
batches = [
    model.pressure_actions(
        298.15, 100000.0, phase="vapor", observables=obs, directions=ds
    )
    for ds in ([t], [p], [t, t], [t, p], [p, p])
]
native = [
    [
        batches[0].molar_density,
        *(b.density_action[0] for b in batches[:2]),
        *(b.density_mixed_action for b in batches[2:]),
    ]
]
native += [
    [batches[0].results[k].value, *(b.results[k].action for b in batches)]
    for k in range(2)
]
physical_increments = (
    1.0,
    temperature_increment_K,
    pressure_increment_Pa,
    temperature_increment_K**2,
    temperature_increment_K * pressure_increment_Pa,
    pressure_increment_Pa**2,
)
native = [
    [entry / increment for entry, increment in zip(row, physical_increments)]
    for row in native
]
rows = []
for row, expected in enumerate(native):
    value, gradient, hessian = fn.evaluate_fgh(
        (298.15, 100000.0, 1.0, text, str(row), "1")
    )
    actual = [value, *gradient[:2], *hessian[:3]]
    assert all(
        (abs(a - b) <= 1e-10 * max(abs(b), 1e-12) for a, b in zip(actual, expected))
    )
    rows.append(
        dict(
            value=value,
            gradient=gradient[:3],
            hessian=hessian[:6],
            maximum_absolute_error=max((abs(a - b) for a, b in zip(actual, expected))),
        )
    )
values = dict(rows=rows, idaes_version=idaes.__version__, record_characters=len(text))
display(
    pd.DataFrame(
        {
            "Quantity": [
                "Density / mol m^-3",
                "Total enthalpy / J mol^-1",
                "ln fugacity coefficient / 1",
            ],
            "Value": [row["value"] for row in rows],
            "Maximum absolute transport error": [
                row["maximum_absolute_error"] for row in rows
            ],
        }
    )
)


epcsaft_property prepared record 1 of at most 16


,Quantity,Value,Maximum absolute transport error
0,Density / mol m^-3,40.415468,0.0
1,Total enthalpy / J mol^-1,-74616.276249,0.0
2,ln fugacity coefficient / 1,-0.001879,0.0


In [4]:
for row, quantity, unit in zip(
    rows,
    ("Density", "Total enthalpy", "ln fugacity coefficient"),
    ("mol m^-3", "J mol^-1", "1"),
):
    gradient = row["gradient"]
    hessian = row["hessian"]
    display(pd.DataFrame([{
        "Quantity": quantity,
        f"T partial / ({unit}) K^-1": gradient[0],
        f"P partial / ({unit}) Pa^-1": gradient[1],
        f"x methane partial / ({unit})": gradient[2],
        f"T,T partial / ({unit}) K^-2": hessian[0],
        f"T,P partial / ({unit}) K^-1 Pa^-1": hessian[1],
        f"P,P partial / ({unit}) Pa^-2": hessian[2],
        f"T,x partial / ({unit}) K^-1": hessian[3],
        f"P,x partial / ({unit}) Pa^-1": hessian[4],
        f"x,x partial / ({unit})": hessian[5],
    }]))
display(pd.DataFrame([{
    "IDAES version": values["idaes_version"],
    "ASL record characters / 1": values["record_characters"],
}]))


,Quantity,T partial / (mol m^-3) K^-1,P partial / (mol m^-3) Pa^-1,x methane partial / (mol m^-3),"T,T partial / (mol m^-3) K^-2","T,P partial / (mol m^-3) K^-1 Pa^-1","P,P partial / (mol m^-3) Pa^-2","T,x partial / (mol m^-3) K^-1","P,x partial / (mol m^-3) Pa^-1","x,x partial / (mol m^-3)"
0,Density,-0.136446,0.000405,0.0,0.000926,-0.000001,1.522167e-11,-0.0,-0.0,-0.0


,Quantity,T partial / (J mol^-1) K^-1,P partial / (J mol^-1) Pa^-1,x methane partial / (J mol^-1),"T,T partial / (J mol^-1) K^-2","T,P partial / (J mol^-1) K^-1 Pa^-1","P,P partial / (J mol^-1) Pa^-2","T,x partial / (J mol^-1) K^-1","P,x partial / (J mol^-1) Pa^-1","x,x partial / (J mol^-1)"
0,Total enthalpy,35.784846,-0.000163,0.0,0.03634,9.387477e-07,-2.174993e-12,0.0,0.0,0.0


,Quantity,T partial / (1) K^-1,P partial / (1) Pa^-1,x methane partial / (1),"T,T partial / (1) K^-2","T,P partial / (1) K^-1 Pa^-1","P,P partial / (1) Pa^-2","T,x partial / (1) K^-1","P,x partial / (1) Pa^-1","x,x partial / (1)"
0,ln fugacity coefficient,0.000022,-1.878545e-08,0.0,-2.742543e-07,2.203641e-10,1.230322e-16,0.0,0.0,0.0


,IDAES version,ASL record characters / 1
0,2.12.0,1590


## Meaning and limits


The comparison verifies value and derivative transport for this neutral
pressure chart. It neither solves a flowsheet nor qualifies ionic or reacting
process derivatives. See the two-liquid settler example for a live IDAES outer
optimization. Thermodynamic accuracy retains the limits of the underlying
parameter and ideal thermal records.
